# Modelagem Baseline — Detecção de Fraudes

Modelos avaliados (sem otimização de hiperparâmetros):  
**Decision Tree · Logistic Regression · Random Forest · XGBoost · LightGBM**

Métricas reportadas para **Treino** e **Validação**:  
Precision · Recall · F1-Score · AUC-ROC · GINI · KS

---
## 1. Imports & configurações

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# imblearn: Pipeline que suporta samplers + SMOTE
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

# ── Estilo global ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
PALETTE = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
RANDOM_STATE = 42

print('Imports OK')

---
## 2. Carga dos dados (Gold layer)

In [ ]:
X = pd.read_parquet('../data/gold/X_train.parquet')
y = pd.read_parquet('../data/gold/y_train.parquet').squeeze()  # Series

print(f'Shape X: {X.shape}')
print(f'Shape y: {y.shape}')
print(f'Taxa de fraude: {y.mean():.4%}')
print(f'\nFraudes  : {y.sum():,}')
print(f'Não-fraude: {(y == 0).sum():,}')

---
## 3. Split treino / validação (estratificado)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Treino  : {X_train.shape[0]:,} amostras  | fraude: {y_train.mean():.4%}')
print(f'Validação: {X_val.shape[0]:,} amostras  | fraude: {y_val.mean():.4%}')

---
## 4. Definição das métricas

In [ ]:
def ks_statistic(y_true, y_prob):
    """Kolmogorov-Smirnov: diferença máxima entre TPR e FPR ao longo dos limiares."""
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return float(np.max(tpr - fpr))


def compute_metrics(model, X_tr, y_tr, X_vl, y_vl, threshold=0.5):
    """
    Retorna dict com métricas de treino e validação.
    threshold: limiar para converter probabilidade em classe.
    """
    results = {}
    for split_name, Xs, ys in [('Treino', X_tr, y_tr), ('Validação', X_vl, y_vl)]:
        prob = model.predict_proba(Xs)[:, 1]
        pred = (prob >= threshold).astype(int)

        auc  = roc_auc_score(ys, prob)
        gini = 2 * auc - 1
        ks   = ks_statistic(ys, prob)

        results[split_name] = {
            'Precision': precision_score(ys, pred, zero_division=0),
            'Recall'   : recall_score(ys, pred, zero_division=0),
            'F1-Score' : f1_score(ys, pred, zero_division=0),
            'AUC-ROC'  : auc,
            'GINI'     : gini,
            'KS'       : ks,
            '_prob'    : prob,   # usado para curvas
            '_pred'    : pred,
            '_y'       : ys,
        }
    return results


print('Funções de métricas definidas.')

---
## 5. Definição dos modelos (SMOTE dentro do Pipeline)

O `imblearn.Pipeline` aplica o SMOTE **apenas durante o `fit`**, nunca na avaliação — garantindo que os dados de validação permaneçam intocados.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)

print(f'Treino original — fraudes: {y_train.sum():,} ({y_train.mean():.4%})')
print('SMOTE será aplicado internamente em cada pipeline durante o fit.')

MODELS = {
    'Decision Tree': Pipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', DecisionTreeClassifier(
            max_depth=8,
            random_state=RANDOM_STATE
        ))
    ]),

    'Logistic Regression': Pipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),

    'Random Forest': Pipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),

    'XGBoost': Pipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            eval_metric='auc',
            use_label_encoder=False,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),

    'LightGBM': Pipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', LGBMClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=-1
        ))
    ]),
}

print('Modelos configurados:', list(MODELS.keys()))

---
## 6. Treinamento & coleta de métricas

In [ ]:
from time import time

all_results = {}  # {model_name: {split: {metric: value}}}

for name, pipeline in MODELS.items():
    t0 = time()
    print(f'Treinando {name}...', end=' ')
    pipeline.fit(X_train, y_train)
    elapsed = time() - t0
    print(f'{elapsed:.1f}s')

    all_results[name] = compute_metrics(
        pipeline, X_train, y_train, X_val, y_val
    )

print('\nTreinamento concluído!')

---
## 7. Tabela de métricas — Treino vs Validação

In [ ]:
METRIC_COLS = ['Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'GINI', 'KS']

rows = []
for model_name, splits in all_results.items():
    for split_name, metrics in splits.items():
        row = {'Modelo': model_name, 'Split': split_name}
        for m in METRIC_COLS:
            row[m] = metrics[m]
        rows.append(row)

df_metrics = pd.DataFrame(rows).set_index(['Modelo', 'Split'])

# Formata como percentual
styled = (
    df_metrics
    .style
    .format('{:.4f}')
    .background_gradient(cmap='RdYlGn', subset=METRIC_COLS, axis=0)
    .set_caption('Métricas Baseline — Treino vs Validação')
    .set_table_styles([{
        'selector': 'caption',
        'props': 'font-size: 14px; font-weight: bold; text-align: left;'
    }])
)

display(styled)

In [ ]:
# Tabela pivotada: linhas = modelo, colunas = (métrica, split)
df_pivot = df_metrics.unstack(level='Split')
df_pivot.columns = [f'{m} [{s}]' for m, s in df_pivot.columns]

# Reordena colunas por métrica
ordered_cols = []
for m in METRIC_COLS:
    ordered_cols += [f'{m} [Treino]', f'{m} [Validação]']
df_pivot = df_pivot[ordered_cols]

print('\n=== Visão Comparativa por Modelo ===')
display(
    df_pivot.style
    .format('{:.4f}')
    .highlight_max(axis=0, color='#c6efce')
    .highlight_min(axis=0, color='#ffc7ce')
    .set_caption('Verde = melhor | Vermelho = pior (por coluna)')
)

---
## 8. Curvas ROC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, split_name in zip(axes, ['Treino', 'Validação']):
    for (model_name, color) in zip(MODELS.keys(), PALETTE):
        m = all_results[model_name][split_name]
        fpr, tpr, _ = roc_curve(m['_y'], m['_prob'])
        auc = m['AUC-ROC']
        ax.plot(fpr, tpr, color=color, lw=2,
                label=f'{model_name} (AUC={auc:.4f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    ax.set_xlabel('False Positive Rate (FPR)')
    ax.set_ylabel('True Positive Rate (TPR)')
    ax.set_title(f'Curva ROC — {split_name}')
    ax.legend(loc='lower right', fontsize=9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

fig.suptitle('Curvas ROC — Baseline', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 9. Curvas Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, split_name in zip(axes, ['Treino', 'Validação']):
    baseline = all_results[list(MODELS.keys())[0]][split_name]['_y'].mean()

    for model_name, color in zip(MODELS.keys(), PALETTE):
        m = all_results[model_name][split_name]
        prec, rec, _ = precision_recall_curve(m['_y'], m['_prob'])
        ap = average_precision_score(m['_y'], m['_prob'])
        ax.plot(rec, prec, color=color, lw=2,
                label=f'{model_name} (AP={ap:.4f})')

    ax.axhline(baseline, color='gray', ls='--', lw=1,
               label=f'Baseline aleatório ({baseline:.3f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'Curva PR — {split_name}')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

fig.suptitle('Curvas Precision-Recall — Baseline', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 10. Curva KS por modelo (Validação)

In [ ]:
n_models = len(MODELS)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, (model_name, color) in enumerate(zip(MODELS.keys(), PALETTE)):
    ax = axes[i]
    m = all_results[model_name]['Validação']
    fpr, tpr, _ = roc_curve(m['_y'], m['_prob'])
    ks_val = m['KS']
    ks_idx = np.argmax(tpr - fpr)

    ax.plot(fpr, tpr, color=color, lw=2, label='TPR')
    ax.plot(fpr, fpr, color='gray', lw=1, ls='--', label='FPR')
    ax.vlines(fpr[ks_idx], fpr[ks_idx], tpr[ks_idx],
              colors='red', lw=2, ls=':', label=f'KS={ks_val:.4f}')
    ax.fill_between(fpr, fpr, tpr, alpha=0.08, color=color)
    ax.set_title(model_name, fontweight='bold')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

# Remove eixos extras
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Curva KS — Validação', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Comparativo de métricas — Gráfico de barras

In [ ]:
# Gráfico comparativo para cada métrica, Treino vs Validação
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

model_names = list(MODELS.keys())
x = np.arange(len(model_names))
width = 0.35

for ax, metric in zip(axes, METRIC_COLS):
    vals_train = [all_results[m]['Treino'][metric] for m in model_names]
    vals_val   = [all_results[m]['Validação'][metric] for m in model_names]

    bars_tr = ax.bar(x - width/2, vals_train, width, label='Treino',
                     color='#2196F3', alpha=0.85, edgecolor='white')
    bars_vl = ax.bar(x + width/2, vals_val,   width, label='Validação',
                     color='#FF5722', alpha=0.85, edgecolor='white')

    # Anota valores
    for bar in bars_tr:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=7.5)
    for bar in bars_vl:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=7.5)

    ax.set_title(metric, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(' ', '\n') for m in model_names], fontsize=9)
    ax.set_ylim(0, min(1.18, max(max(vals_train), max(vals_val)) * 1.25))
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

fig.suptitle('Comparativo de Métricas — Treino vs Validação', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 12. Matrizes de Confusão (Validação)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, (model_name, color) in enumerate(zip(MODELS.keys(), PALETTE)):
    ax = axes[i]
    m = all_results[model_name]['Validação']
    cm = confusion_matrix(m['_y'], m['_pred'])

    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Legítima', 'Fraude'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(model_name, fontweight='bold')
    ax.set_xlabel('Predito'); ax.set_ylabel('Real')

# Remove eixo extra
axes[-1].set_visible(False)

fig.suptitle('Matrizes de Confusão — Validação (threshold=0.5)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 13. Radar Chart — Validação

In [ ]:
from matplotlib.patches import FancyArrowPatch

metrics_radar = METRIC_COLS
N = len(metrics_radar)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # fechar o polígono

fig, ax = plt.subplots(figsize=(8, 8),
                       subplot_kw=dict(polar=True))

for (model_name, color) in zip(MODELS.keys(), PALETTE):
    values = [all_results[model_name]['Validação'][m] for m in metrics_radar]
    values += values[:1]
    ax.plot(angles, values, color=color, lw=2, label=model_name)
    ax.fill(angles, values, color=color, alpha=0.08)

ax.set_thetagrids(np.degrees(angles[:-1]), metrics_radar, fontsize=11)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
ax.set_title('Radar de Métricas — Validação', fontsize=13,
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)

plt.tight_layout()
plt.show()

---
## 14. Distribuição de probabilidades (score) — Validação

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, (model_name, color) in enumerate(zip(MODELS.keys(), PALETTE)):
    ax = axes[i]
    m = all_results[model_name]['Validação']
    prob = np.array(m['_prob'])
    y_true = np.array(m['_y'])

    ax.hist(prob[y_true == 0], bins=60, density=True,
            alpha=0.6, color='#2196F3', label='Legítima')
    ax.hist(prob[y_true == 1], bins=60, density=True,
            alpha=0.6, color='#F44336', label='Fraude')
    ax.axvline(0.5, color='black', ls='--', lw=1.5, label='threshold=0.5')
    ax.set_title(model_name, fontweight='bold')
    ax.set_xlabel('Score (probabilidade de fraude)')
    ax.set_ylabel('Densidade')
    ax.legend(fontsize=9)

axes[-1].set_visible(False)

fig.suptitle('Distribuição de Score por Classe — Validação',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 15. Overfitting — Delta Treino × Validação

In [ ]:
delta_rows = []
for model_name in MODELS.keys():
    row = {'Modelo': model_name}
    for metric in METRIC_COLS:
        tr = all_results[model_name]['Treino'][metric]
        vl = all_results[model_name]['Validação'][metric]
        row[f'Δ {metric}'] = tr - vl   # positivo = overfitting
    delta_rows.append(row)

df_delta = pd.DataFrame(delta_rows).set_index('Modelo')

display(
    df_delta.style
    .format('{:+.4f}')
    .background_gradient(cmap='RdYlGn_r', axis=0)
    .set_caption('Δ = Treino − Validação (positivo = overfitting)')
    .set_table_styles([{
        'selector': 'caption',
        'props': 'font-size: 13px; font-weight: bold; text-align: left;'
    }])
)

# Heatmap visual
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(df_delta, annot=True, fmt='+.3f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, center=0)
ax.set_title('Heatmap de Overfitting (Δ Treino − Validação)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 16. Importância de Features (modelos tree-based)

In [ ]:
tree_models = ['Decision Tree', 'Random Forest', 'XGBoost', 'LightGBM']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, model_name in zip(axes, tree_models):
    pipeline = MODELS[model_name]
    # O último step é sempre 'model'
    clf = pipeline.named_steps['model']
    importances = clf.feature_importances_
    feature_names = X_train.columns.tolist()

    df_imp = pd.Series(importances, index=feature_names).sort_values(ascending=True)
    top15 = df_imp.tail(15)

    color = PALETTE[['Decision Tree','Random Forest','XGBoost','LightGBM'].index(model_name)]
    top15.plot(kind='barh', ax=ax, color=color, edgecolor='white')
    ax.set_title(f'{model_name} — Top 15 Features', fontweight='bold')
    ax.set_xlabel('Importância')
    ax.set_ylabel('')
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

fig.suptitle('Importância de Features — Modelos Tree-Based', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 17. Resumo Final

In [ ]:
print('=' * 65)
print('RESUMO FINAL — BASELINE (sem otimização de hiperparâmetros)')
print('=' * 65)

# Melhor modelo por AUC-ROC na validação
best_auc = max(MODELS.keys(), key=lambda m: all_results[m]['Validação']['AUC-ROC'])
best_f1  = max(MODELS.keys(), key=lambda m: all_results[m]['Validação']['F1-Score'])
best_ks  = max(MODELS.keys(), key=lambda m: all_results[m]['Validação']['KS'])

print(f'\nMelhor AUC-ROC (val): {best_auc}  '
      f"→ {all_results[best_auc]['Validação']['AUC-ROC']:.4f}")
print(f'Melhor F1-Score (val): {best_f1}   '
      f"→ {all_results[best_f1]['Validação']['F1-Score']:.4f}")
print(f'Melhor KS (val):       {best_ks}   '
      f"→ {all_results[best_ks]['Validação']['KS']:.4f}")

print('\n--- Validação completa dos modelos ---')
for m in METRIC_COLS:
    scores = {name: all_results[name]['Validação'][m] for name in MODELS}
    best   = max(scores, key=scores.get)
    print(f'  {m:12s}: {best} ({scores[best]:.4f})')

print('\nPróximos passos sugeridos:')
print('  1. Otimização de hiperparâmetros (Optuna/GridSearch) nos top modelos')
print('  2. Ajuste de threshold (maximizar F1-Score ou Recall mínimo)')
print('  3. Calibração de probabilidades (Isotonic / Platt)')
print('  4. Ensemble / Stacking entre os melhores candidatos')